In [36]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from collections import defaultdict, Counter

sns.set_style("whitegrid")

import matplotlib

# set font size to 16
matplotlib.rcParams.update({"font.size": 16})

SEED = 4

In [37]:
n_runs = 100
# ["BENIGN", "Portmap", "NetBIOS", "LDAP", "MSSQL", "UDP", "UDPLag", "Syn"]
start_type = "UDP"

#Use the csv files again
#Load accuracy, precision, recall and f1 score
def load_dict_from_csv(filename):
    df = pd.read_csv(filename)
    nested_dict = {}
    for (run, method), group in df.groupby(['Run', 'Method']):
        nested_dict.setdefault(run, {})[method] = group['Value'].tolist()
    return nested_dict

#Load time measurements
def load_time_dict_from_csv(filename):
    df = pd.read_csv(filename)
    time_dict = {}
    for (run, method), group in df.groupby(['Run', 'Method']):
        pairs = group[['Selection_Time', 'Train_Time']].values.tolist()
        time_dict.setdefault(run, {})[method] = pairs
    return time_dict

#Load type distributions
def load_distributions_dict_from_csv(filename, all_labels):
    df = pd.read_csv(filename)
    dist_dict = {}
    for (run, method), group in df.groupby(['Run', 'Method']):
        arrs = group[all_labels].values.tolist()
        dist_dict.setdefault(run, {})[method] = arrs
    return dist_dict

#Load benchmark values
def load_benchmark_dict_from_csv(filename):
    df = pd.read_csv(filename)
    return {col: df[col].dropna().tolist() for col in df.columns}

def get_summary(loaded_dict):
    methods = list(loaded_dict[0].keys())
    summary_dict = {}
    for method in methods:
        all_runs = np.array([loaded_dict[run][method] for run in range(n_runs)]) # shape: (100, array_length)
        # Compute statistics
        mean_values = np.mean(all_runs, axis=0)
        plow_values = np.percentile(all_runs, 10, axis=0)
        phigh_values = np.percentile(all_runs, 90, axis=0)
        # Store in nested structure for easy plotting
        summary_dict[method] = {
            "mean": mean_values.tolist(),
            "plow": plow_values.tolist(),
            "phigh": phigh_values.tolist()
        }
    return summary_dict

benchmark_dict_loaded = load_benchmark_dict_from_csv(f"./results_dict_new/benchmark_dict_{start_type}_{n_runs}.csv")
accuracy_dict_loaded = load_dict_from_csv( f"./results_dict_new/accuracy_dict_{start_type}_{n_runs}.csv")
precision_dict_loaded = load_dict_from_csv( f"./results_dict_new/precision_dict_{start_type}_{n_runs}.csv")
recall_dict_loaded = load_dict_from_csv( f"./results_dict_new/recall_dict_{start_type}_{n_runs}.csv")
f1score_dict_loaded = load_dict_from_csv( f"./results_dict_new/f1score_dict_{start_type}_{n_runs}.csv")
time_dict_loaded = load_time_dict_from_csv(f"./results_dict_new/time_dict_{start_type}_{n_runs}.csv")
all_labels = ["BENIGN", "Portmap", "NetBIOS", "LDAP", "MSSQL", "UDP", "UDPLag", "Syn"]
distributions_dict_loaded = load_distributions_dict_from_csv(f"./results_dict_new/distributions_dict_{start_type}_{n_runs}.csv", all_labels)

# Mean Benchmark Values
benchmark_means_dict = {k: np.mean(v) for k, v in benchmark_dict_loaded.items()}
#Mean Accuracy Values
accuracy_summary = get_summary(accuracy_dict_loaded)
# Mean Precision Values
precision_summary = get_summary(precision_dict_loaded)
# Mean Recall Values
recall_summary =  get_summary(recall_dict_loaded)
# Mean F1 Score Values
f1score_summary = get_summary(f1score_dict_loaded)



In [38]:
#Calculate distributions at specific f1 score thresholds
f1score_dict_loaded = load_dict_from_csv( f"./results_dict/f1score_dict_{start_type}_{n_runs}.csv")
results = {}
for run in range(len(f1score_dict_loaded)):
    for method in f1score_dict_loaded[run].keys():
        if len(f1score_dict_loaded[run][method]) == 351:
            f1score_dict_loaded[run][method] = np.array(f1score_dict_loaded[run][method])[::10]
            f1score_dict_loaded[run][method] = np.array(f1score_dict_loaded[run][method])[1:]
        else:
            f1score_dict_loaded[run][method] = np.array(f1score_dict_loaded[run][method])[1:]

for run in range(n_runs):
    run_data = f1score_dict_loaded[run]
    methods = list(run_data.keys())

    # Flatten per method into (method, index, value)
    f1_events = []
    for method in methods:
        values = np.array(run_data[method])
        for i, v in enumerate(values):
            f1_events.append((method, i, v))
    
    # Sort events by iteration (index)
    f1_events.sort(key=lambda x: x[1])
    
    # Track first >=0.95 and first/second >=0.99
    first_95 = None
    first_99 = None
    second_99 = None
    third_99 = None
    seen_99_methods = set()
    
    for method, idx, val in f1_events:
        if first_95 is None and val >= 0.95:
            first_95 = (method, idx)
        if val >= 0.99 and method not in seen_99_methods:
            seen_99_methods.add(method)
            if first_99 is None:
                first_99 = (method, idx)
            elif second_99 is None:
                second_99 = (method, idx)
            elif third_99 is None:
                third_99 = (method, idx)
        if first_95 and first_99 and second_99 and third_99:
            break

    results[run] = {
        "first_95_method": first_95[0] if first_95 else None,
        "first_95_index": first_95[1] if first_95 else None,
        "first_99_method": first_99[0] if first_99 else None,
        "first_99_index": first_99[1] if first_99 else None,
        "second_99_method": second_99[0] if second_99 else None,
        "second_99_index": second_99[1] if second_99 else None,
        "third_99_index": third_99[1] if third_99 else None,
        "third_99_method": third_99[0] if third_99 else None
    }


# Ensure consistent method list
methods = list(distributions_dict_loaded[0].keys())
# Containers for distributions at each threshold
all_distr_95 = {method: [] for method in methods}
all_distr_99_1 = {method: [] for method in methods}
all_distr_99_2 = {method: [] for method in methods}
all_distr_99_3 = {method: [] for method in methods}

for run in range(n_runs):
    idx_1 = results[run]["first_95_index"]
    idx_2 = results[run]["first_99_index"]
    idx_3 = results[run]["second_99_index"]
    idx_4 = results[run]["third_99_index"]

    for method in methods:
        dist_arr = np.array(distributions_dict_loaded[run][method])
        n_points = dist_arr.shape[0]

        # Determine whether method is batch or single-step
        if n_points == 350:
            i_1 = ((idx_1 + 1) * 10) - 1
            i_2 = ((idx_2 + 1) * 10) - 1
            i_3 = ((idx_3 + 1) * 10) - 1 
            i_4 = ((idx_4 + 1) * 10) - 1
        else:
            i_1 = idx_1
            i_2 = idx_2
            i_3 = idx_3
            i_4 = idx_4

        all_distr_95[method].append(dist_arr[i_1])
        all_distr_99_1[method].append(dist_arr[i_2])
        all_distr_99_2[method].append(dist_arr[i_3])
        all_distr_99_3[method].append(dist_arr[i_4])

# Mean Distributions
mean_distr_95 = {m: np.mean(all_distr_95[m], axis=0) for m in methods}
mean_distr_99_1 = {m: np.mean(all_distr_99_1[m], axis=0) for m in methods}
mean_distr_99_2 = {m: np.mean(all_distr_99_2[m], axis=0) for m in methods}
mean_distr_99_3 = {m: np.mean(all_distr_99_3[m], axis=0) for m in methods}

In [39]:
#Get the mean sorting
results_summary = {}
f1score_summary_mean = defaultdict(lambda: None)
for method in f1score_summary.keys():
    if len(f1score_summary[method]["mean"]) == 351:
        f1score_summary_mean[method]= np.array(f1score_summary[method]["mean"])[::10]
        f1score_summary_mean[method] = np.array(f1score_summary_mean[method])[1:]
    else:
        f1score_summary_mean[method] = np.array(f1score_summary[method]["mean"])[1:]



methods = list(f1score_summary_mean.keys())

# Flatten per method into (method, index, value)
f1_events = []
for method in methods:
    values = np.array(f1score_summary_mean[method])
    for i, v in enumerate(values):
        f1_events.append((method, i, v))
    
# Sort events by iteration (index)
f1_events.sort(key=lambda x: x[1])
    
# Track first >=0.95 and first/second >=0.99
first_95 = None
first_99 = None
second_99 = None
third_99 = None
seen_99_methods = set()
    
for method, idx, val in f1_events:
    if first_95 is None and val >= 0.95:
        first_95 = (method, idx)
    if val >= 0.99 and method not in seen_99_methods:
        seen_99_methods.add(method)
        if first_99 is None:
            first_99 = (method, idx)
        elif second_99 is None:
            second_99 = (method, idx)
        elif third_99 is None:
            third_99 = (method, idx)
    if first_95 and first_99 and second_99 and third_99:
        break

results_summary = {
    "first_95_method": first_95[0] if first_95 else None,
    "first_95_index": first_95[1] if first_95 else None,
    "first_99_method": first_99[0] if first_99 else None,
    "first_99_index": first_99[1] if first_99 else None,
    "second_99_method": second_99[0] if second_99 else None,
    "second_99_index": second_99[1] if second_99 else None,
    "third_99_index": third_99[1] if third_99 else None,
    "third_99_method": third_99[0] if third_99 else None
}



In [40]:
"""
def plot_mean_distribution(mean_dict, title, extra_info = None):
    # Grouped as: [ [Benign%, Malicious%], ... ]
    methods = ["KU Batch", "KU Single", "Predict Proba Batch", "Predict Proba Single", "QBC", "Random"]
    #categories = ["Portmap", "NetBIOS", "LDAP", "MSSQL", "UDP", "UDPLag", "Syn"]
    categories = ["Portmap", "NetBIOS", "LDAP", "MSSQL", "UDP", "Syn"]

    # Collect data in the same order as `methods`
    data = np.array([
        mean_dict["KU_batch"],
        mean_dict["KU_single"],
        mean_dict["Predictproba_batch"],
        mean_dict["Predictproba_single"],
        mean_dict["QBC"],
        mean_dict["Random"]
    ])

    x = np.arange(len(methods))
    width = 0.1

    data = np.delete(data, [0, 6], axis = 1) #remove benign and UDPLag

    plt.figure(figsize=(10, 6))
    for i, category in enumerate(categories):
        plt.bar(x + (i - len(categories)/2) * width, data[:, i], width, label=category)
    
    if extra_info is not None:
        method_name, idx = extra_info
        plt.plot([], [], ' ', label=f"Reached by: {method_name}, added samples: {(idx+1)*10}")

    plt.xticks(x, methods, rotation=15, fontsize = 10)
    plt.ylabel("Percentages")
    plt.xlabel("Algorithms")
    plt.title(title)
    plt.ylim(0, 30)
    plt.legend(fontsize=8, loc='upper right', ncol=2) 
    plt.grid(axis="y", linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.show()

"""

def plot_mean_distribution(mean_dict, title, extra_info=None):
    methods = ["KU Batch", "KU Single", "Predict Proba Batch", "Predict Proba Single", "QBC", "Random"]
    categories = ["Benign","Portmap", "NetBIOS", "LDAP", "MSSQL", "UDP", "Syn"]

    data = np.array([
        mean_dict["KU_batch"],
        mean_dict["KU_single"],
        mean_dict["Predictproba_batch"],
        mean_dict["Predictproba_single"],
        mean_dict["QBC"],
        mean_dict["Random"]
    ])

    data = np.delete(data, 6, axis=1)  # remove Benign + UDPLag

    x = np.arange(len(methods))
    width = 0.1
    plt.figure(figsize=(10, 6))

    bar_handles = []
    # Draw category bars
    for i, category in enumerate(categories):
        bars = plt.bar(x + (i - len(categories)/2) * width, data[:, i], width, label=category)
        bar_handles.append(bars[0])

    # Main legend (categories only)
    category_legend = plt.legend(handles=bar_handles,
                                 labels=categories,
                                 fontsize=8,
                                 loc='upper right',
                                 ncol=2,
                                 title="Attack Types")

    plt.gca().add_artist(category_legend)  # keep the first legend

    # Extra info legend (separate)
    if extra_info is not None:
        method_name, idx = extra_info
        extra_label = f"Reached by: {method_name}\nAdded samples: {(idx+1)*10}"

        plt.legend(
            handles=[plt.Line2D([0], [0], color='white')],  # invisible handle
            labels=[extra_label],
            loc="upper center",
            frameon=True,
            fontsize=9,
        )

    plt.xticks(x, methods, rotation=15, fontsize=10)
    plt.ylabel("Percentages")
    plt.xlabel("Algorithms")
    plt.title(title)
    plt.ylim(0, 80)
    plt.grid(axis="y", linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.show()



# Plot each mean distribution set separately
"""
plot_mean_distribution(mean_distr_95, 
                       f"Mean Distribution at First 95% F1 Score (Start: {start_type}, {n_runs} runs)",
                       extra_info=(results_summary["first_95_method"], results_summary["first_95_index"]))

plot_mean_distribution(mean_distr_99_1, 
                       f"Mean Distribution at First 99% F1 Score (Start: {start_type}, {n_runs} runs)",
                       extra_info=(results_summary["first_99_method"], results_summary["first_99_index"]))

plot_mean_distribution(mean_distr_99_2, 
                       f"Mean Distribution at Second 99% F1 Score (Start: {start_type}, {n_runs} runs)",
                       extra_info=(results_summary["second_99_method"], results_summary["second_99_index"]))

plot_mean_distribution(mean_distr_99_3, 
                       f"Mean Distribution at Third 99% F1 Score (Start: {start_type}, {n_runs} runs)",
                       extra_info=(results_summary["third_99_method"], results_summary["third_99_index"]))
#Compute for fixed length
"""
"""
for siz in datasize:
    plot_mean_distribution(mean_distr_len[siz], 
                            f"Mean Distribution at Train Size Length {siz} (Start: {start_type}, {n_runs} runs)",
                            extra_info=None)
"""

'\nfor siz in datasize:\n    plot_mean_distribution(mean_distr_len[siz], \n                            f"Mean Distribution at Train Size Length {siz} (Start: {start_type}, {n_runs} runs)",\n                            extra_info=None)\n'

In [41]:

EPS = 1e-8
attack_types = ["Benign", "Portmap", "NetBIOS", "LDAP", "MSSQL", "UDP", "UDPLag", "Syn"]

def normalize(p):
    p = np.asarray(p, dtype=float)
    return p / p.sum()

def kl_contributions(p, q):
    """
    Per-class contribution to KL(P || Q)
    """
    #p = normalize(p)
    #q = normalize(q)
    q = np.clip(q, EPS, None)
    return p * np.log(p / q)


In [42]:
matrices = {
    "First_95": mean_distr_95,
    "First_99": mean_distr_99_1,
    "Second_99": mean_distr_99_2,
    "Third_99": mean_distr_99_3
}

rows_diff = []

for stage, distr_dict in matrices.items():
    # Get the Random distribution for this stage as the baseline
    p_random = distr_dict["Random"]

    for method, p_method in distr_dict.items():
        if method == "Random":
            continue

        # Calculate simple difference (Algorithm - Random)
        # Result is in percentage points (e.g., +5.0 means 5% more than random)
        diffs = (p_method - p_random)/p_random * 100.0

        for attack, value in zip(attack_types, diffs):
            rows_diff.append({
                "Stage": stage,
                "Algorithm": method,
                "Attack_Type": attack,
                "Percentage_Difference": value
            })

# Create DataFrame
df_diff = pd.DataFrame(rows_diff)

# Save to CSV
csv_path_diff = f"./perc_diff/{start_type}_percentage_difference_vs_random.csv"
df_diff.to_csv(csv_path_diff, index=False)

print(f"✅ Saved Percentage Difference table to: {csv_path_diff}")
print(df_diff.head())

✅ Saved Percentage Difference table to: ./perc_diff/UDP_percentage_difference_vs_random.csv
      Stage Algorithm Attack_Type  Percentage_Difference
0  First_95  KU_batch      Benign              28.683199
1  First_95  KU_batch     Portmap             183.000928
2  First_95  KU_batch     NetBIOS             264.012945
3  First_95  KU_batch        LDAP             -19.318415
4  First_95  KU_batch       MSSQL             -67.224457


/tmp/ipykernel_679331/3036716119.py:20: RuntimeWarning: invalid value encountered in divide
  diffs = (p_method - p_random)/p_random * 100.0
/tmp/ipykernel_679331/3036716119.py:20: RuntimeWarning: divide by zero encountered in divide
  diffs = (p_method - p_random)/p_random * 100.0
